# USDA Corn Yield Prediction with Machine Learning

## Project Objective

Develop and evaluate machine-learning models
to predict annual corn yield (bushels per acre)
using historical USDA agricultural data.

Models:
- Linear Regression
- Random Forest Regression
- Gradient Boosting Regression

Technologies:
Python, PySpark, Databricks, scikit-learn, MLflow

Data Source:
USDA NASS historical corn-yield observations, 2020–2025.

## Reproducibility and data limitations

- **Source**: Historical USDA NASS corn yields, 2020–2025. Notebook expects a populated Databricks Delta table named `workspace.usda_training.silver_usda_corn` (246 state-year observations, 41 state categories). This table is **not included** in the notebook export; configure or recreate it before running.
- **Requirements**: Databricks with Spark/PySpark, pandas, numpy, matplotlib, scikit-learn, and MLflow. Notebook uses `display()` and Databricks workspace file paths.
- **Evaluation**: 2023/2024 are exploratory, chronological development-validation years. Multiple models were also inspected on 2025; consequently **2025 is not an untouched independent final test**.
- **Data leakage control**: Lagged features are computed within state and only from earlier years. Comparisons of one/two-feature models use identical eligible records.
- **Interpretation**: Limited six-year dataset; results are educational research, not operational agricultural forecasts. State-level arithmetic averages are not national production-weighted estimates.
- **Side effects**: Model tracking cells create new MLflow runs when rerun. Report-writing cells write to `/Workspace/Shared/usda_ml_evaluation_report.csv` and overwrite that report path. Historical Delta source table is read-only in this notebook.



In [ ]:

from pyspark.sql import functions as F

print("USDA MACHINE LEARNING PROJECT")

print("Spark version:", spark.version)

print("PYSPARK ENVIRONMENT READY")


In [ ]:

# Historical USDA data — read only
silver_table = "workspace.usda_training.silver_usda_corn"

df_usda = spark.table(silver_table)

print("Historical USDA table loaded")
print("Total records:", df_usda.count())

print("\nAvailable columns:")
df_usda.printSchema()

print("USDA HISTORICAL DATA LOADED")


In [ ]:

display(df_usda.limit(10))

print("USDA DATA PREVIEW PASSED")


In [ ]:

from pyspark.sql import functions as F

# Check for available reporting years
year_summary = (
    df_usda
    .groupBy("year")
    .agg(
        F.count("*").alias("records")
    )
    .orderBy("year")
)

display(year_summary)

print("USDA YEAR COVERAGE CHECK PASSED")


In [ ]:

from pyspark.sql import functions as F

print("Total records:", df_usda.count())

print("\nColumn names:")
print(df_usda.columns)

print("\nRecords by year:")
df_usda.groupBy("year").count().orderBy("year").show()

print("\nMissing values by column:")
df_usda.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_usda.columns
]).show()

print("USDA HISTORICAL DATA INSPECTION COMPLETED")


In [ ]:

from pyspark.sql import functions as F

# 1. Identify duplicate state-year combinations
duplicate_keys = (
    df_usda
    .groupBy("year", "state")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_keys.count()

# 2. Identify invalid yield values
invalid_yields = df_usda.filter(
    (F.col("yield_bu_acre") <= 0) |
    F.isnan(F.col("yield_bu_acre").cast("double")) |
    F.col("yield_bu_acre").isNull()
)

invalid_count = invalid_yields.count()

# 3. Check the available years
years = sorted(
    row["year"]
    for row in df_usda.select("year").distinct().collect()
)

# 4. Check the number of unique states
state_count = df_usda.select("state").distinct().count()

print("Total records:", df_usda.count())
print("Unique states:", state_count)
print("Duplicate state-year keys:", duplicate_count)
print("Invalid yields:", invalid_count)
print("Available years:", years)

assert df_usda.count() == 246
assert state_count == 41
assert duplicate_count == 0
assert invalid_count == 0
assert years == [2020, 2021, 2022, 2023, 2024, 2025]

print("USDA ML DATA QUALITY VALIDATION PASSED")


In [ ]:

from pyspark.sql import functions as F

print("USDA CORN YIELD — DESCRIPTIVE STATISTICS")

yield_stats = df_usda.select(
    F.count("*").alias("records"),
    F.round(F.avg("yield_bu_acre"), 2).alias("average_yield"),
    F.round(F.min("yield_bu_acre"), 2).alias("minimum_yield"),
    F.round(F.max("yield_bu_acre"), 2).alias("maximum_yield"),
    F.round(F.stddev("yield_bu_acre"), 2).alias("standard_deviation")
)

display(yield_stats)

print("USDA DESCRIPTIVE STATISTICS COMPLETED")


In [ ]:

yearly_yield = (
    df_usda
    .groupBy("year")
    .agg(
        F.round(
            F.avg("yield_bu_acre"), 2
        ).alias("average_yield")
    )
    .orderBy("year")
)

display(yearly_yield)

print("USDA YEARLY YIELD ANALYSIS COMPLETED")


In [ ]:

import matplotlib.pyplot as plt

yield_pd = yearly_yield.toPandas()

plt.figure(figsize=(10, 5))

plt.plot(
    yield_pd["year"],
    yield_pd["average_yield"],
    marker="o",
    linewidth=2
)

plt.title("Average USDA Corn Yield by Year (2020–2025)")
plt.xlabel("Year")
plt.ylabel("Average Yield (Bushels per Acre)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("USDA YIELD TREND VISUALIZATION COMPLETED")


In [ ]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Define chronological ordering within each state
state_window = Window.partitionBy("state").orderBy("year")

# Create previous-year yield feature
df_features = (
    df_usda
    .withColumn(
        "previous_year_yield",
        F.lag("yield_bu_acre", 1).over(state_window)
    )
)

print("FEATURE ENGINEERING RESULTS")

display(
    df_features.select(
        "state",
        "year",
        "yield_bu_acre",
        "previous_year_yield"
    ).orderBy("state", "year")
)

print("USDA FEATURE ENGINEERING COMPLETED")


In [ ]:

df_ml = df_features.filter(
    F.col("previous_year_yield").isNotNull()
)

print("Original records:", df_usda.count())
print("Machine-learning records:", df_ml.count())

df_ml.groupBy("year").count().orderBy("year").show()

assert df_ml.count() == 205

print("USDA ML FEATURE DATASET READY")


In [ ]:

# Chronological train/test split
train_df = df_ml.filter(F.col("year") <= 2024)
test_df = df_ml.filter(F.col("year") == 2025)

print("Training records:", train_df.count())
print("Testing records:", test_df.count())

print("\nTraining years:")
train_df.select("year").distinct().orderBy("year").show()

print("\nTesting years:")
test_df.select("year").distinct().orderBy("year").show()

assert train_df.count() == 164
assert test_df.count() == 41

print("USDA TRAIN TEST SPLIT PASSED")


In [ ]:

from sklearn.linear_model import LinearRegression
import numpy as np

# Convert the small datasets to pandas
train_pd = train_df.select(
    "previous_year_yield",
    "yield_bu_acre"
).toPandas()

test_pd = test_df.select(
    "state",
    "year",
    "previous_year_yield",
    "yield_bu_acre"
).toPandas()

# Define model inputs and targets
X_train = train_pd[["previous_year_yield"]]
y_train = train_pd["yield_bu_acre"]

X_test = test_pd[["previous_year_yield"]]
y_test = test_pd["yield_bu_acre"]

# Train the model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict 2025 corn yields
y_pred = lr_model.predict(X_test)

print("Linear Regression training completed")
print("Coefficient:", round(lr_model.coef_[0], 4))
print("Intercept:", round(lr_model.intercept_, 4))
print("Predictions generated:", len(y_pred))

assert len(y_pred) == 41

print("USDA LINEAR REGRESSION MODEL TRAINED")


In [ ]:

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Model metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Baseline: predict the previous year's yield
baseline_pred = X_test["previous_year_yield"].to_numpy()

baseline_mae = mean_absolute_error(
    y_test, baseline_pred
)
baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_pred)
)

print("2025 MODEL EVALUATION")
print("----------------------------")
print("Linear Regression MAE:", round(mae, 2))
print("Linear Regression RMSE:", round(rmse, 2))
print("Linear Regression R²:", round(r2, 4))

print("\nPREVIOUS-YEAR BASELINE")
print("Baseline MAE:", round(baseline_mae, 2))
print("Baseline RMSE:", round(baseline_rmse, 2))

print("\nUSDA MODEL EVALUATION COMPLETED")


In [ ]:

import pandas as pd
import numpy as np

# Create a table of 2025 predictions
prediction_results = test_pd[
    ["state", "year", "previous_year_yield", "yield_bu_acre"]
].copy()

prediction_results["predicted_yield"] = y_pred

prediction_results["absolute_error"] = np.abs(
    prediction_results["yield_bu_acre"]
    - prediction_results["predicted_yield"]
)

prediction_results = prediction_results.sort_values(
    "absolute_error",
    ascending=False
)

print("2025 CORN YIELD PREDICTION RESULTS")

display(prediction_results.round(2))

assert len(prediction_results) == 41

print("USDA PREDICTION COMPARISON PASSED")


In [ ]:

import matplotlib.pyplot as plt

actual = prediction_results["yield_bu_acre"]
predicted = prediction_results["predicted_yield"]

plt.figure(figsize=(9, 6))

plt.scatter(
    actual,
    predicted,
    alpha=0.75,
    s=65
)

# Perfect-prediction reference line
low = min(actual.min(), predicted.min())
high = max(actual.max(), predicted.max())

plt.plot(
    [low, high],
    [low, high],
    linestyle="--",
    color="red",
    label="Perfect prediction"
)

plt.xlabel("Actual Yield (Bushels per Acre)")
plt.ylabel("Predicted Yield (Bushels per Acre)")
plt.title("2025 USDA Corn Yield: Actual vs Predicted")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("USDA ACTUAL VS PREDICTED CHART COMPLETED")


In [ ]:

# Identify the five largest prediction errors

top_errors = prediction_results[
    [
        "state",
        "yield_bu_acre",
        "predicted_yield",
        "absolute_error"
    ]
].sort_values(
    "absolute_error",
    ascending=False
).head(5)

print("TOP 5 LARGEST PREDICTION ERRORS")

display(top_errors.round(2))

print("USDA ERROR ANALYSIS COMPLETED")


In [ ]:

from sklearn.ensemble import RandomForestRegressor

# Train on the same 2021-2024 data
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=4,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Predict the held-out 2025 yields
rf_pred = rf_model.predict(X_test)

print("Random Forest training completed")
print("Number of trees:", rf_model.n_estimators)
print("Predictions generated:", len(rf_pred))

assert len(rf_pred) == 41

print("USDA RANDOM FOREST MODEL TRAINED")


In [ ]:

rf_mae = mean_absolute_error(y_test, rf_pred)

rf_rmse = np.sqrt(
    mean_squared_error(y_test, rf_pred)
)

rf_r2 = r2_score(y_test, rf_pred)

print("2025 RANDOM FOREST EVALUATION")
print("--------------------------------")

print("Random Forest MAE:", round(rf_mae, 2))
print("Random Forest RMSE:", round(rf_rmse, 2))
print("Random Forest R²:", round(rf_r2, 4))

print("\nUSDA RANDOM FOREST EVALUATION COMPLETED")


In [ ]:

comparison = pd.DataFrame({
    "Model": [
        "Previous-Year Baseline",
        "Linear Regression",
        "Random Forest"
    ],
    "MAE": [
        baseline_mae,
        mae,
        rf_mae
    ],
    "RMSE": [
        baseline_rmse,
        rmse,
        rf_rmse
    ],
    "R2": [
        r2_score(y_test, baseline_pred),
        r2,
        rf_r2
    ]
})

comparison = comparison.sort_values("MAE")

display(comparison.round(4))

print("USDA MODEL COMPARISON COMPLETED")


In [ ]:

from pyspark.sql import functions as F

# Load the original historical USDA data
df_usda = spark.table(
    "workspace.usda_training.silver_usda_corn"
)

# Recreate the previous-year yield feature
from pyspark.sql.window import Window

state_window = Window.partitionBy("state").orderBy("year")

df_ml = (
    df_usda
    .withColumn(
        "previous_year_yield",
        F.lag("yield_bu_acre", 1).over(state_window)
    )
    .filter(F.col("previous_year_yield").isNotNull())
)

# Recreate the chronological split
train_df = df_ml.filter(F.col("year") <= 2024)
test_df = df_ml.filter(F.col("year") == 2025)

# Convert to pandas
train_pd = train_df.select(
    "previous_year_yield",
    "yield_bu_acre"
).toPandas()

test_pd = test_df.select(
    "state",
    "year",
    "previous_year_yield",
    "yield_bu_acre"
).toPandas()

# Restore machine-learning variables
X_train = train_pd[["previous_year_yield"]]
y_train = train_pd["yield_bu_acre"]

X_test = test_pd[["previous_year_yield"]]
y_test = test_pd["yield_bu_acre"]

# Verify the datasets
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

assert len(X_train) == 164
assert len(X_test) == 41

print("USDA ML TRAINING VARIABLES RESTORED")


In [ ]:

from sklearn.ensemble import GradientBoostingRegressor

# Train Gradient Boosting using the same data
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,
    min_samples_leaf=5,
    random_state=42
)

gb_model.fit(X_train, y_train)

# Predict 2025 corn yields
gb_pred = gb_model.predict(X_test)

print("Gradient Boosting training completed")
print("Number of trees:", gb_model.n_estimators)
print("Learning rate:", gb_model.learning_rate)
print("Predictions generated:", len(gb_pred))

assert len(gb_pred) == 41

print("USDA GRADIENT BOOSTING MODEL TRAINED")


In [ ]:

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

# Evaluate Gradient Boosting predictions
gb_mae = mean_absolute_error(y_test, gb_pred)

gb_rmse = np.sqrt(
    mean_squared_error(y_test, gb_pred)
)

gb_r2 = r2_score(y_test, gb_pred)

print("2025 GRADIENT BOOSTING EVALUATION")
print("--------------------------------")

print("Gradient Boosting MAE:", round(gb_mae, 2))
print("Gradient Boosting RMSE:", round(gb_rmse, 2))
print("Gradient Boosting R²:", round(gb_r2, 4))

print("\nUSDA GRADIENT BOOSTING EVALUATION COMPLETED")


In [ ]:

import mlflow
import sklearn

print("MLflow version:", mlflow.__version__)
print("scikit-learn version:", sklearn.__version__)

print("USDA MLFLOW ENVIRONMENT READY")


In [ ]:

import mlflow

experiment = mlflow.get_experiment_by_name(
    mlflow.get_experiment(
        mlflow.active_run().info.experiment_id
    ).name
) if mlflow.active_run() else None

print(
    "Active MLflow run:",
    mlflow.active_run() is not None
)

print("MLFLOW INITIAL CHECK COMPLETED")


In [ ]:

import mlflow

# Record the completed Gradient Boosting experiment
with mlflow.start_run(
    run_name="USDA_Gradient_Boosting_2025"
) as run:

    # Record model parameters
    mlflow.log_param("model", "GradientBoostingRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 2)
    mlflow.log_param("min_samples_leaf", 5)
    mlflow.log_param("random_state", 42)

    # Record evaluation metrics
    mlflow.log_metric("MAE", gb_mae)
    mlflow.log_metric("RMSE", gb_rmse)
    mlflow.log_metric("R2", gb_r2)

    # Record experiment details
    mlflow.set_tag("dataset", "USDA historical corn yield")
    mlflow.set_tag("training_years", "2021-2024")
    mlflow.set_tag("test_year", "2025")
    mlflow.set_tag("test_records", "41")
    mlflow.set_tag("feature", "previous_year_yield")
    mlflow.set_tag("evaluation", "exploratory_holdout")

    run_id = run.info.run_id

print("MLflow experiment recorded")
print("Run ID:", run_id)
print("MAE:", round(gb_mae, 2))
print("RMSE:", round(gb_rmse, 2))
print("R2:", round(gb_r2, 4))

print("USDA MLFLOW EXPERIMENT LOGGED")


In [ ]:

import mlflow
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Verify training and testing data are available
assert len(X_train) == 164
assert len(X_test) == 41

# Recreate Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

# Recreate Random Forest using original settings
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=4,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

# Previous-year baseline
baseline_pred = X_test["previous_year_yield"].to_numpy()

models = [
    (
        "Previous_Year_Baseline",
        baseline_pred,
        {"method": "previous_year_yield"}
    ),
    (
        "Linear_Regression",
        lr_pred,
        {"model": "LinearRegression"}
    ),
    (
        "Random_Forest",
        rf_pred,
        {
            "model": "RandomForestRegressor",
            "n_estimators": 100,
            "max_depth": 4,
            "min_samples_leaf": 5,
            "random_state": 42
        }
    )
]

for name, predictions, parameters in models:

    model_mae = mean_absolute_error(
        y_test, predictions
    )

    model_rmse = np.sqrt(
        mean_squared_error(y_test, predictions)
    )

    model_r2 = r2_score(
        y_test, predictions
    )

    with mlflow.start_run(
        run_name="USDA_" + name + "_2025"
    ) as run:

        mlflow.log_params(parameters)

        mlflow.log_metrics({
            "MAE": model_mae,
            "RMSE": model_rmse,
            "R2": model_r2
        })

        mlflow.set_tags({
            "dataset": "USDA historical corn yield",
            "training_years": "2021-2024",
            "test_year": "2025",
            "test_records": "41",
            "feature": "previous_year_yield",
            "evaluation": "exploratory_holdout"
        })

    print(
        name,
        "| MAE:", round(model_mae, 2),
        "| RMSE:", round(model_rmse, 2),
        "| R2:", round(model_r2, 4)
    )

print("USDA ALL FOUR MODELS LOGGED")


In [ ]:

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

validation_results = []

for test_year in [2023, 2024]:

    # Train only on years before the test year
    fold_train = df_ml.filter(
        F.col("year") < test_year
    )

    fold_test = df_ml.filter(
        F.col("year") == test_year
    )

    train_pd_fold = fold_train.select(
        "previous_year_yield", "yield_bu_acre"
    ).toPandas()

    test_pd_fold = fold_test.select(
        "previous_year_yield", "yield_bu_acre"
    ).toPandas()

    X_fold_train = train_pd_fold[["previous_year_yield"]]
    y_fold_train = train_pd_fold["yield_bu_acre"]

    X_fold_test = test_pd_fold[["previous_year_yield"]]
    y_fold_test = test_pd_fold["yield_bu_acre"]

    # Fresh model for each fold
    fold_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=4,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    fold_model.fit(X_fold_train, y_fold_train)

    fold_pred = fold_model.predict(X_fold_test)

    fold_mae = mean_absolute_error(
        y_fold_test, fold_pred
    )

    fold_rmse = np.sqrt(
        mean_squared_error(y_fold_test, fold_pred)
    )

    # Simple baseline for comparison
    baseline_pred_fold = X_fold_test[
        "previous_year_yield"
    ].to_numpy()

    baseline_mae_fold = mean_absolute_error(
        y_fold_test, baseline_pred_fold
    )

    validation_results.append({
        "test_year": test_year,
        "training_records": len(X_fold_train),
        "testing_records": len(X_fold_test),
        "RF_MAE": round(fold_mae, 2),
        "RF_RMSE": round(fold_rmse, 2),
        "Baseline_MAE": round(baseline_mae_fold, 2)
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

print("USDA WALK-FORWARD VALIDATION COMPLETED")


In [ ]:

print("RANDOM FOREST WALK-FORWARD SUMMARY")

print(
    "Average RF MAE:",
    round(validation_df["RF_MAE"].mean(), 2)
)

print(
    "Average Baseline MAE:",
    round(validation_df["Baseline_MAE"].mean(), 2)
)

print(
    "RF better in:",
    int(
        (validation_df["RF_MAE"] <
         validation_df["Baseline_MAE"]).sum()
    ),
    "of",
    len(validation_df),
    "validation years"
)

print("USDA CHRONOLOGICAL VALIDATION SUMMARY COMPLETED")


In [ ]:

import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.metrics import mean_absolute_error

all_results = []

for test_year in [2023, 2024]:

    train = df_ml.filter(
        F.col("year") < test_year
    ).select(
        "previous_year_yield", "yield_bu_acre"
    ).toPandas()

    test = df_ml.filter(
        F.col("year") == test_year
    ).select(
        "previous_year_yield", "yield_bu_acre"
    ).toPandas()

    X_tr = train[["previous_year_yield"]]
    y_tr = train["yield_bu_acre"]

    X_te = test[["previous_year_yield"]]
    y_te = test["yield_bu_acre"]

    # Previous-year baseline
    baseline_predictions = X_te[
        "previous_year_yield"
    ].to_numpy()

    all_results.append({
        "test_year": test_year,
        "model": "Previous-Year Baseline",
        "MAE": mean_absolute_error(
            y_te, baseline_predictions
        )
    })

    models = {
        "Linear Regression": LinearRegression(),

        "Random Forest": RandomForestRegressor(
            n_estimators=100,
            max_depth=4,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        ),

        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=2,
            min_samples_leaf=5,
            random_state=42
        )
    }

    for name, model in models.items():
        model.fit(X_tr, y_tr)
        predictions = model.predict(X_te)

        all_results.append({
            "test_year": test_year,
            "model": name,
            "MAE": mean_absolute_error(
                y_te, predictions
            )
        })

validation_all = pd.DataFrame(all_results)

print("CHRONOLOGICAL MODEL VALIDATION RESULTS")
display(
    validation_all.sort_values(
        ["test_year", "model"]
    ).round(2)
)

print("USDA MULTI-MODEL VALIDATION COMPLETED")


In [ ]:

model_summary = (
    validation_all
    .groupby("model", as_index=False)
    .agg(
        Average_MAE=("MAE", "mean"),
        Worst_Year_MAE=("MAE", "max")
    )
    .sort_values("Average_MAE")
)

print("AVERAGE VALIDATION PERFORMANCE")

display(model_summary.round(2))

print(
    "Lowest validation MAE:",
    model_summary.iloc[0]["model"]
)

print("USDA MODEL VALIDATION COMPARISON COMPLETED")


In [ ]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

state_window = Window.partitionBy("state").orderBy("year")

df_features_v2 = (
    df_usda
    .withColumn(
        "previous_year_yield",
        F.lag("yield_bu_acre", 1).over(state_window)
    )
    .withColumn(
        "two_years_ago_yield",
        F.lag("yield_bu_acre", 2).over(state_window)
    )
    .withColumn(
        "two_year_average",
        (
            F.col("previous_year_yield")
            + F.col("two_years_ago_yield")
        ) / 2.0
    )
    .filter(
        F.col("two_year_average").isNotNull()
    )
)

print("FEATURE ENGINEERING V2")

print("Records:", df_features_v2.count())

df_features_v2.groupBy("year").count().orderBy("year").show()

display(
    df_features_v2.select(
        "state",
        "year",
        "yield_bu_acre",
        "previous_year_yield",
        "two_years_ago_yield",
        "two_year_average"
    ).orderBy("state", "year").limit(12)
)

print("USDA FEATURE ENGINEERING V2 COMPLETED")


In [ ]:

# Verify the engineered dataset
record_count = df_features_v2.count()

duplicate_count = (
    df_features_v2
    .groupBy("year", "state")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_features = df_features_v2.filter(
    F.col("previous_year_yield").isNull()
    | F.col("two_years_ago_yield").isNull()
    | F.col("two_year_average").isNull()
).count()

print("Total feature records:", record_count)
print("Duplicate state-year keys:", duplicate_count)
print("Missing engineered features:", invalid_features)

assert record_count == 164
assert duplicate_count == 0
assert invalid_features == 0

print("USDA FEATURE ENGINEERING V2 VALIDATION PASSED")


In [ ]:

import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.metrics import mean_absolute_error
from pyspark.sql import functions as F

feature_results = []

feature_sets = {
    "One Feature": ["previous_year_yield"],
    "Two Features": [
        "previous_year_yield",
        "two_year_average"
    ]
}

for test_year in [2023, 2024]:

    # Use the same eligible V2 records for both
    train = (
        df_features_v2
        .filter(F.col("year") < test_year)
        .toPandas()
    )

    test = (
        df_features_v2
        .filter(F.col("year") == test_year)
        .toPandas()
    )

    # Verify chronological separation
    assert train["year"].max() < test_year
    assert len(test) == 41

    y_train_fold = train["yield_bu_acre"]
    y_test_fold = test["yield_bu_acre"]

    for feature_name, columns in feature_sets.items():

        X_train_fold = train[columns]
        X_test_fold = test[columns]

        models = {
            "Linear Regression": LinearRegression(),

            "Random Forest": RandomForestRegressor(
                n_estimators=100,
                max_depth=4,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=-1
            ),

            "Gradient Boosting": GradientBoostingRegressor(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=2,
                min_samples_leaf=5,
                random_state=42
            )
        }

        for model_name, model in models.items():

            model.fit(X_train_fold, y_train_fold)

            predictions = model.predict(X_test_fold)

            feature_results.append({
                "Test_Year": test_year,
                "Features": feature_name,
                "Model": model_name,
                "Train_Records": len(train),
                "Test_Records": len(test),
                "MAE": mean_absolute_error(
                    y_test_fold, predictions
                )
            })

feature_comparison = pd.DataFrame(feature_results)

print("ONE-FEATURE VS TWO-FEATURE COMPARISON")

display(
    feature_comparison.sort_values(
        ["Test_Year", "Model", "Features"]
    ).round(2)
)

print("USDA FEATURE COMPARISON COMPLETED")


In [ ]:

feature_summary = (
    feature_comparison
    .groupby(["Model", "Features"], as_index=False)
    .agg(
        Average_MAE=("MAE", "mean"),
        Worst_Year_MAE=("MAE", "max")
    )
    .sort_values(["Model", "Features"])
)

print("AVERAGE FEATURE COMPARISON")

display(feature_summary.round(2))

print("USDA FEATURE PERFORMANCE SUMMARY COMPLETED")


In [ ]:

from sklearn.linear_model import LinearRegression
import pandas as pd

feature_columns = [
    "previous_year_yield",
    "two_year_average"
]

# Train on historical data through 2024
development_data = (
    df_features_v2
    .filter(F.col("year") <= 2024)
    .toPandas()
)

X_development = development_data[feature_columns]
y_development = development_data["yield_bu_acre"]

feature_model = LinearRegression()
feature_model.fit(X_development, y_development)

coefficients = pd.DataFrame({
    "Feature": feature_columns,
    "Coefficient": feature_model.coef_
})

print("LINEAR REGRESSION FEATURE COEFFICIENTS")

display(coefficients.round(4))

print("Intercept:", round(feature_model.intercept_, 4))
print("Training records:", len(X_development))

print("USDA FEATURE COEFFICIENT ANALYSIS COMPLETED")


In [ ]:

correlation_matrix = development_data[
    [
        "previous_year_yield",
        "two_year_average",
        "yield_bu_acre"
    ]
].corr()

print("USDA FEATURE CORRELATION MATRIX")

display(correlation_matrix.round(3))

print("USDA FEATURE CORRELATION ANALYSIS COMPLETED")


In [ ]:

import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from pyspark.sql import functions as F

feature_columns = [
    "previous_year_yield",
    "two_year_average"
]

# Train on 2022-2023
diagnostic_train = (
    df_features_v2
    .filter(F.col("year") <= 2023)
    .toPandas()
)

# Validate on 2024
diagnostic_test = (
    df_features_v2
    .filter(F.col("year") == 2024)
    .toPandas()
)

diagnostic_model = LinearRegression()

diagnostic_model.fit(
    diagnostic_train[feature_columns],
    diagnostic_train["yield_bu_acre"]
)

predictions_2024 = diagnostic_model.predict(
    diagnostic_test[feature_columns]
)

residual_results = diagnostic_test[
    ["state", "year", "yield_bu_acre"]
].copy()

residual_results["predicted_yield"] = predictions_2024

residual_results["residual"] = (
    residual_results["yield_bu_acre"]
    - residual_results["predicted_yield"]
)

residual_results["absolute_error"] = (
    residual_results["residual"].abs()
)

print("2024 RESIDUAL ANALYSIS")

print("Training records:", len(diagnostic_train))
print("Validation records:", len(diagnostic_test))

display(
    residual_results.sort_values(
        "absolute_error", ascending=False
    ).round(2)
)

print("USDA RESIDUAL DATASET CREATED")


In [ ]:

print("2024 MODEL DIAGNOSTICS")
print("----------------------------")

print(
    "MAE:",
    round(mean_absolute_error(
        residual_results["yield_bu_acre"],
        residual_results["predicted_yield"]
    ), 2)
)

print(
    "RMSE:",
    round(np.sqrt(mean_squared_error(
        residual_results["yield_bu_acre"],
        residual_results["predicted_yield"]
    )), 2)
)

print(
    "Mean residual:",
    round(residual_results["residual"].mean(), 2)
)

print(
    "Maximum absolute error:",
    round(residual_results["absolute_error"].max(), 2)
)

print(
    "Underpredictions:",
    int((residual_results["residual"] > 0).sum())
)

print(
    "Overpredictions:",
    int((residual_results["residual"] < 0).sum())
)

print("USDA RESIDUAL STATISTICS COMPLETED")


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.scatter(
    residual_results["predicted_yield"],
    residual_results["residual"],
    alpha=0.75
)

plt.axhline(
    y=0,
    color="red",
    linestyle="--",
    label="Zero prediction error"
)

plt.xlabel("Predicted Yield (Bushels per Acre)")
plt.ylabel("Residual (Bushels per Acre)")
plt.title("USDA Corn Yield Model Residuals — 2024")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("USDA RESIDUAL VISUALIZATION COMPLETED")


In [ ]:

import mlflow
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

actual = residual_results["yield_bu_acre"]
predicted = residual_results["predicted_yield"]

validation_mae = mean_absolute_error(actual, predicted)
validation_rmse = np.sqrt(
    mean_squared_error(actual, predicted)
)
validation_r2 = r2_score(actual, predicted)

with mlflow.start_run(
    run_name="USDA_Linear_Regression_Two_Features_2024"
) as run:

    mlflow.log_params({
        "model": "LinearRegression",
        "feature_count": 2,
        "feature_1": "previous_year_yield",
        "feature_2": "two_year_average",
        "training_years": "2022-2023",
        "validation_year": 2024
    })

    mlflow.log_metrics({
        "MAE": validation_mae,
        "RMSE": validation_rmse,
        "R2": validation_r2,
        "mean_residual": float(
            residual_results["residual"].mean()
        ),
        "max_absolute_error": float(
            residual_results["absolute_error"].max()
        )
    })

    mlflow.set_tags({
        "dataset": "USDA historical corn yield",
        "evaluation": "development_validation",
        "validation_records": "41",
        "data_source": "USDA NASS historical data",
        "data_leakage_control": "chronological_split"
    })

    validation_run_id = run.info.run_id

print("MLflow validation experiment recorded")
print("Run ID:", validation_run_id)
print("MAE:", round(validation_mae, 2))
print("RMSE:", round(validation_rmse, 2))
print("R2:", round(validation_r2, 4))

print("USDA TWO-FEATURE VALIDATION LOGGED")


In [ ]:

import pandas as pd

# Collect the results from earlier lessons
report_data = {
    "Project": "USDA Corn Yield Prediction",
    "Data Source": "USDA NASS historical corn yields",
    "Historical Years": "2020-2025",
    "Historical Records": 246,
    "States": 41,
    "Initial ML Records": 205,
    "Enhanced Feature Records": 164,
    "Validation Years": "2023, 2024",
    "Features": [
        "previous_year_yield",
        "two_year_average"
    ],
    "Best Development Model": "Linear Regression",
    "Best Average Development MAE": 15.35,
    "2024 Validation MAE": 16.68,
    "2024 Validation RMSE": 20.96,
    "2024 Validation R2": 0.6939,
    "MLflow Tracking": "Enabled",
    "Final Independent Test": "Not performed"
}

report_df = pd.DataFrame(
    [
        {
            "Metric": key,
            "Value": str(value)
        }
        for key, value in report_data.items()
    ]
)

print("USDA MACHINE LEARNING PROJECT SUMMARY")

display(report_df)

print("USDA MODEL EVALUATION REPORT GENERATED")


In [ ]:

# Save the summary within the Databricks workspace
report_path = "/Workspace/Shared/usda_ml_evaluation_report.csv"

report_df.to_csv(
    report_path,
    index=False
)

print("Report saved:", report_path)

print("USDA MODEL REPORT SAVED")


In [ ]:

# Reload the saved report
saved_report = pd.read_csv(
    "/Workspace/Shared/usda_ml_evaluation_report.csv"
)

print("Saved report entries:", len(saved_report))

display(saved_report)

assert len(saved_report) == len(report_data)

print("USDA MODEL REPORT VERIFIED")
